In [1]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
import json
import collections
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import zipfile

from tqdm import tqdm
import shutil

np.random.seed(42)

In [2]:
import warnings
warnings.filterwarnings('ignore')

### Verify GPU

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_type = torch.float32 if device.type == "cuda" else torch.float16
device, torch_type

(device(type='cuda'), torch.float32)

In [4]:
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090
Memory: 25.43 GB


### Loading package

In [5]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [6]:
from py.utils import verifyDir,verifyFile, verifyDataFrame

In [7]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/media/felipe/DATA19/datasets/', '/media/felipe/DATA19/models/')

In [8]:
OUT_NAME = f"{cfg.PERCEPTION_METRIC}"
OUT_NAME += "_filter" if cfg.FILTER_FEATURES else ""
OUT_NAME += "_bin" if cfg.BINARIZE_FEATURES else ""
OUT_NAME

'safety'

In [9]:
CF_DIR = f"{cfg.MODEL_PATH}"
CF_DIR += f"{cfg.SEG_DATASET}_{cfg.UPD_DATASET}" if cfg.USE_UPD else f"{cfg.SEG_DATASET}"
CF_DIR += "_group/" if cfg.BY_GROUPS else "/"
CF_DIR += "" if cfg.USE_UPD else f"{cfg.SEG_MODEL_NAME}/"
CF_DIR += f"counterfactuals/{OUT_NAME}/"
CF_DIR

'/media/felipe/DATA19/models/ADE20k_UPD4k_group/counterfactuals/safety/'

### Loading Data

In [10]:
cf_test_data_df = pd.read_csv(f"{CF_DIR}cf_test_data.csv", sep=";", low_memory=False)

In [11]:
features_name = cf_test_data_df.iloc[:, 1:-2].columns.tolist()

# LLM Interpretations

In [12]:
from py.counterfactuals import CounterfactualAnalyzer

In [13]:
cf_analyzer = CounterfactualAnalyzer()
cf_analyzer.load(CF_DIR)

In [14]:
results = cf_analyzer.get_results()
unsafe2safe_df = results["nearest_cf_variation"].copy()
unsafe2safe_df["diff_new_prob"] = unsafe2safe_df["diff_prob"].apply(lambda x: x[0][1])
unsafe2safe_df.sort_values(by="diff_new_prob", ascending=False, inplace=True)
unsafe2safe_df.head(30)

,city_elements,construction,floor,human,sky,terrain_vehicle,vegetation,broken_damaged_bricks_wall,broken_damaged_pavement_road,broken_window,...,trashcan,image_id,orig_prob,new_prob,diff_prob,orig_class,new_class,desired_class_prob,euclidean_dist,diff_new_prob
254,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,-13.240833,0.000000,0.0,...,0.0,50f5eba3fdc9f065f00083b4,"[[0.786811507936508, 0.213188492063492]]","[[0.07651785714285714, 0.9234821428571428]]","[[-0.7102936507936508, 0.7102936507936508]]",0,1,0.923482,13.250270,0.710294
96,0.0,0.0,0.0,0.000000,-8.621667,0.0,0.0,-6.120833,0.000000,0.0,...,0.0,50f5ec42fdc9f065f00088dd,"[[0.8395955988455988, 0.1604044011544011]]","[[0.24554166666666674, 0.7544583333333332]]","[[-0.594053932178932, 0.5940539321789321]]",0,1,0.754458,10.573445,0.594054
84,0.0,0.0,0.0,0.287500,0.000000,0.0,0.0,0.000000,0.000000,0.0,...,0.0,50f5ebd0fdc9f065f00085cf,"[[0.709109126984127, 0.2908908730158731]]","[[0.11582539682539685, 0.8841746031746034]]","[[-0.5932837301587301, 0.5932837301587304]]",0,1,0.884175,9.348589,0.593284
641,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,-9.331667,0.000000,0.0,...,0.0,50f5ebcefdc9f065f0008595,"[[0.8593353174603177, 0.14066468253968256]]","[[0.26763275613275617, 0.7323672438672436]]","[[-0.5917025613275615, 0.591702561327561]]",0,1,0.732367,9.350936,0.591703
407,0.0,0.0,0.0,0.300000,0.000000,0.0,0.0,-14.436667,0.000000,0.0,...,0.0,50f5eb42fdc9f065f000812a,"[[0.9367142857142856, 0.06328571428571429]]","[[0.35732142857142857, 0.6426785714285714]]","[[-0.579392857142857, 0.5793928571428572]]",0,1,0.642679,14.439783,0.579393
277,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,-13.897500,0.000000,0.0,...,0.0,50f5ec18fdc9f065f0008711,"[[0.7359415584415585, 0.26405844155844166]]","[[0.17390692640692637, 0.8260930735930735]]","[[-0.5620346320346321, 0.5620346320346319]]",0,1,0.826093,13.900738,0.562035
398,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,-4.676667,0.000000,0.0,...,0.0,50f5eb67fdc9f065f0008295,"[[0.9167180735930738, 0.0832819264069264]]","[[0.3728190836940836, 0.6271809163059164]]","[[-0.5438989898989901, 0.54389898989899]]",0,1,0.627181,4.676667,0.543899
48,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,-5.753333,0.000000,0.0,...,0.0,50f5ec3efdc9f065f0008897,"[[0.8271726190476191, 0.17282738095238095]]","[[0.28430339105339114, 0.7156966089466091]]","[[-0.542869227994228, 0.5428692279942282]]",0,1,0.715697,5.775019,0.542869
323,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,-3.847500,0.000000,0.0,...,0.0,50f5ec3ffdc9f065f00088a9,"[[0.8498035714285712, 0.15019642857142854]]","[[0.3203315295815296, 0.6796684704184703]]","[[-0.5294720418470416, 0.5294720418470418]]",0,1,0.679668,3.894003,0.529472
170,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,-5.845833,0.000000,0.0,...,0.0,50f5eb20fdc9f065f0008008,"[[0.7670714285714284, 0.23292857142857146]]","[[0.25845833333333335, 0.7415416666666664]]","[[-0.5086130952380951, 0.508613095238095]]",0,1,0.741542,5.867177,0.508613


### Prompt

In [15]:
from py.LLM import Chat, QuantizedChat

In [16]:
# llm_chat = QuantizedChat(model_name="Qwen/Qwen2.5-3B-Instruct")
llm_chat = Chat(model_name="Qwen/Qwen2.5-3B-Instruct")

Loading model: Qwen/Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully on cuda


In [17]:
llm_chat.set_system_message(
    "You are an expert in urban planning and visual perception.\n" \
    "Your task is to explain how changes in specific visual elements of a street scene might influence the way people perceive safety."
)

In [18]:
%%time
llm_interpretations = []

for index, row_ in tqdm(unsafe2safe_df.iterrows()):
    changes = row_[features_name].to_frame().T
    # increase
    #added = changes.gt(0).apply(lambda r: r.index[r].tolist(), axis=1)
    added = changes.columns[(changes > 0).any() & (changes > cfg.CHANGE_THRESHOLD).any()].tolist()
    increased = [ f"{a_}: increased in { round( changes[[a_]].values[0][0] , 3) }" for a_ in added ]
    # decrease
    #substracted = changes.lt(0).apply(lambda r: r.index[r].tolist(), axis=1)
    substracted = changes.columns[(changes < 0).any() & (changes < -1*cfg.CHANGE_THRESHOLD).any()].tolist()
    decreased = [ f"{a_}: decreased in { round( abs(changes[[a_]].values[0][0]) , 3) }" for a_ in substracted ]

    prompt_changes = f"""
    We have a list of visual elements added or substracted from an image.
    These changes come from comparing the original image with a generated counterfactual version.
    [Added elements]:
    -{chr(10).join(increased) if increased else "None"}
    [Removed elements]:
    -{chr(10).join(decreased) if decreased else "None"}
    Please write a concise explanation (2–4 sentences) describing the elements to add and remove and how these changes may affect safety perception.
    """

    # Get response from the model
    response = llm_chat.chat(prompt_changes, temperature=0.7, max_new_tokens=256)
    
    # Store the result
    llm_interpretations.append({
        'index': index,
        'increased': increased,
        'decreased': decreased,
        'explanation': response
    })
    
    print(f"\n--- Row {index} ---")
    print(f"Response: {response}\n")
    break

0it [00:01, ?it/s]


--- Row 254 ---
Response: The addition of more car taxies could enhance perceived safety by providing additional eyes on the street, which can deter potential crime and increase vigilance among pedestrians. Conversely, removing broken and damaged brick walls decreases hazards that might cause slips and falls, thereby improving overall safety perceptions.

CPU times: user 861 ms, sys: 204 ms, total: 1.06 s
Wall time: 1.1 s
